In [2]:
from typing import List

In [ ]:
"""
695 Max Area of Island
给定一个二维的0-1矩阵，其中0表示海洋，1表示陆地。
单独的或相邻的陆地可以形成岛屿，每个格子只与其上下左右四个格子相邻。求最大的岛屿面积。
"""
# 输入是一个二维数组，输出是一个整数，表示最大的岛屿面积

# 栈写法
def maxAreaOfIsland(grid: List[List[int]]) -> int:
    direction = [-1, 0, 1, 0, -1]
    m, n, max_area = len(grid), len(grid[0]), 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                island = []
                # 初始化第一个节点
                local_area = 1
                grid[i][j] = 0
                island.append((i, j))
                # DFS
                while len(island) > 0:
                    r, c = island.pop()
                    for k in range(4):
                        x, y = r + direction[k], c + direction[k+1]
                        #放入满足条件的相邻节点
                        if 0 <= x < m and 0 <= y < n and grid[x][y] == 1:
                            local_area += 1
                            grid[x][y] = 0
                            island.append((x, y))
                max_area = max(max_area, local_area)
    return max_area


```markdown
## 一、题目再翻译

给定一个 `m×n` 的 0-1 矩阵：

- **0** 表示海洋（不能走）  
- **1** 表示陆地（可以走，只与上下左右四个方向相邻）

**岛屿** = 一片连通的 `1`。  
求：所有岛屿中，包含 `1` 的个数最多的那个岛屿的面积。  
如果没有 `1`，返回 `0`。

---

## 二、宏观思路

1. 遍历每个格子 `(i, j)`。  
2. 一旦发现 `grid[i][j] == 1`，就说明“发现了一座新岛屿的起点”。  
3. 从该起点出发，把整座岛屿“翻”成 `0`（俗称“沉岛”或“染色”），同时统计本次翻掉的 `1` 的个数，就是该岛面积。  
4. 用全局变量 `max_area` 记录历史最大值。

“翻岛”过程可以用：

- DFS（递归或显式栈）  
- BFS（队列）  
- 并查集（Union-Find，少见，但也能做）

你给出的代码就是“**显式栈模拟 DFS**”版本。

---

## 三、代码逐行解剖

```python
direction = [-1, 0, 1, 0, -1]
```

技巧：前四个与后四个错位，正好组成 `(dx, dy)` 四元组：

| k | dx | dy | 方向 |
|---|----|----|------|
| 0 | -1 | 0  | 上   |
| 1 | 0  | 1  | 右   |
| 2 | 1  | 0  | 下   |
| 3 | 0  | -1 | 左   |

```python
m, n = len(grid), len(grid[0])
max_area = 0
```

- `m` 行 `n` 列，`max_area` 存答案。

```python
for i in range(m):
    for j in range(n):
        if grid[i][j] == 1:          # 发现新岛屿
            island = []              # 手动栈，存放“待访问”的陆地坐标
            local_area = 1           # 先把当前格子算进去
            grid[i][j] = 0           # 立即标记成 0，防止回头又扫到
            island.append((i, j))    # 起点入栈
```

```python
            while island:            # 标准 DFS：栈不空就继续
                r, c = island.pop()  # 弹出一个节点
                for k in range(4):   # 四个方向
                    x = r + direction[k]
                    y = c + direction[k+1]
                    if 0 <= x < m and 0 <= y < n and grid[x][y] == 1:
                        local_area += 1
                        grid[x][y] = 0      # 沉岛
                        island.append((x, y))  # 新陆地入栈
```

注意：

- 这里 `pop()` 默认从列表尾部弹，相当于栈，所以是**深度优先**。
- 每次发现新 `1` 立即置 `0`，保证每个 `1` 只被统计一次，全局无重复。

```python
            max_area = max(max_area, local_area)
return max_area
```

一座岛处理完，更新答案。

---

## 四、复杂度

| 维度 | 复杂度 | 说明 |
|------|--------|------|
| 时间 | `O(m×n)` | 每个格子最多被访问一次 |
| 空间 | `O(m×n)` | 最坏情况下栈里同时存下一条“蛇形”岛屿 |

- 若用递归版 DFS，系统栈深度也是 `O(m×n)`，LeetCode 测试例一般不会爆栈；若担心可改成 BFS 队列，效果一样。

---

## 五、常见疑问 / 易错点

1. **为什么一定要 `grid[i][j]=0`？**  
   防止同一座岛被多次统计，也防止死循环（栈里反复压入相同坐标）。

2. **方向数组写成 `[[-1,0],[0,1],[1,0],[0,-1]]` 可以吗？**  
   当然可以，而且可读性更好。作者那种“错位”写法只是炫技，节省 4 行代码。

3. **如果题目要求“不能修改原矩阵”怎么办？**  
   额外开一个 `m×n` 的 `visited` 数组记录是否访问过；或者把访问过的 `1` 改成 `2`，最后可以再改回来。

4. **可以用 BFS 吗？**  
   把 `island` 列表换成 `collections.deque`，`popleft()` 就是广度优先。复杂度同阶。

```

In [3]:
# 递归写法

# 辅函数
def dfs(grid: List[List[int]], r: int, c: int) -> int:
    if r < 0 or r >= len(grid) or c < 0 or c >= len(grid[0]) or grid[r][c] == 0:
        return 0
    grid[r][c] = 0
    return (1 + dfs(grid, r + 1, c) + dfs(grid, r - 1, c) +
            dfs(grid, r, c + 1) + dfs(grid, r, c -1))
# 主函数
def maxAreaOfIsland(grid: List[List[int]]) -> int:
    max_area = 0
    for i in range(len(grid)):
        for j in range(len(grid[0])):
            max_area = max(max_area, dfs(grid, i, j))
    return max_area

```markdown
## 一、整体思路

1. 主函数 `maxAreaOfIsland` 逐格扫描矩阵。  
2. 一旦发现 `grid[i][j] == 1`，就以此点为起点调用辅函数 `dfs`。  
3. `dfs` 的职责：  
   - 把当前能连通的所有 `1` 全部“沉”成 `0`  
   - 同时返回这片岛屿的面积（即 `1` 的个数）  
4. 主函数用 `max_area` 记录历史最大值，扫描结束即得答案。

---

## 二、辅函数 `dfs` 逐行讲解

```python
def dfs(grid, r, c):
```

- **入参**  
  - `grid`：原矩阵（会被原地修改，避免重复访问）  
  - `r, c`：当前正在访问的坐标  

```python
    if r < 0 or r >= len(grid) or c < 0 or c >= len(grid[0]) or grid[r][c] == 0:
        return 0
```

- **递归出口**（“撞墙”或“海水”）  
  1. 越界：行号或列号跑到矩阵外面 → 面积 `0`  
  2. 当前格子是 `0` → 要么本来就是海，要么已被别人沉岛 → 面积 `0`  
  满足任一条件立即返回 `0`，不再继续扩散。

```python
    grid[r][c] = 0
```

- **沉岛**：把当前 `1` 改成 `0`，保证每个陆地只被统计一次，同时防止死循环。

```python
    return (1
            + dfs(grid, r + 1, c)
            + dfs(grid, r - 1, c)
            + dfs(grid, r, c + 1)
            + dfs(grid, r, c - 1))
```

- **分治统计**  
  - 当前的 `1` 贡献面积 `1`  
  - 向上下左右四个方向继续“扫荡”，把返回的面积累加  
  - 最终得到以 `(r,c)` 为起点的整座岛屿面积

---

## 三、主函数 `maxAreaOfIsland` 逐行讲解

```python
def maxAreaOfIsland(grid):
    max_area = 0
```

- `max_area` 用来记录“目前为止看到的最大岛屿”。

```python
    for i in range(len(grid)):
        for j in range(len(grid[0])):
```

- 双重循环枚举每个格子，寻找“尚未被沉岛的 `1`”。

```python
            max_area = max(max_area, dfs(grid, i, j))
```

- 一旦找到 `1`，就把整片岛屿面积算出来，并更新全局最大值。  
- 由于 `dfs` 内部会把访问过的 `1` 全部清零，后续扫描不会重复计算。

```python
    return max_area
```

- 扫描结束，返回最大岛屿面积。

---

## 四、复杂度分析

| 维度 | 复杂度 | 说明 |
|------|--------|------|
| 时间 | `O(m × n)` | 每个格子最多被访问一次，且一旦被置 `0` 不再重复 |
| 空间 | `O(m × n)` | 递归深度最坏等于整张矩阵都是 `1`，系统调用栈可能达到 `m × n` 层 |

> 若担心最坏情况爆栈，可改用显式栈（DFS 迭代版）或队列（BFS），复杂度同阶。

---

## 五、易错点 & 常见疑问

1. **为什么不先判断 `grid[r][c] == 1` 再进递归？**  
   主函数已经保证只在 `1` 的位置调用 `dfs`；而递归内部再次遇到 `0` 会直接返回，逻辑正确。

2. **沉岛时直接改 `grid` 会不会影响原数据？**  
   题目通常允许原地修改；若不允许，可额外维护一个 `visited` 矩阵。

3. **方向写死四个 `dfs` 调用，能不能用循环？**  
   可以，把四个偏移量放进数组 `dirs = [(1,0),(-1,0),(0,1),(0,-1)]`，然后循环调用即可：

   ```python
   area = 1
   for dr, dc in dirs:
       area += dfs(grid, r + dr, c + dc)
   return area
   ```

4. **递归太深会爆栈吗？**  
   - Python 默认递归深度 ≈ 1000，LeetCode 官方测试例一般不会超过。  
   - 若矩阵很大且全是 `1`，深度可达 `10^4` 以上，建议用迭代版 DFS/BFS。

---

## 六、一行总结

> **“遇到 1 就炸平整座岛，边炸边数格子。”**  
把这套递归模板背下来，岛屿类问题（数量、面积、周长、形状）都能快速改写。
```

In [4]:
from typing import Set

In [ ]:
"""
547 Number of Provinces
给定一个二维的0-1矩阵，如果第(i,j)位置是1，则表示第i个城市和第j个城市处于同一个城市圈。
已知城市的相邻关系是可以传递的，
即如果a和b相邻，b和c相邻，那么a和c也相邻，
换言之这三个城市处于同一个城市圈之内。求一共有多少个城市圈。
"""

# 输入是一个二维数组，输出是一个整数，表示城市圈数量。
# 因为城市相邻关系具有对称性，该二维数组为对称矩阵。
# 同时，因为自己也处于自己的城市圈，对角线上的值全部为1。

def dfs(isConnected: List[List[int]], city: int, visited: Set[int]):
    # 首先将该city放入visited集合里
    visited.add(city)
    # 对所有的节点进行遍历
    for i in range(len(isConnected)):
        # 如果从该城市到目标城市有连接 且 目标城市还没有被遍历
        if isConnected[city][i] == 1 and i not in visited:
            # 对目标城市继续进行广度优先搜索
            dfs(isConnected, i, visited)

def findCircleNum(isConnected: List[List[int]]) -> int:
    # 初始化计数和已经探索过的地点集
    count = 0
    visited = set()
    # 对每个节点进行遍历
    for i in range(len(isConnected)):
        # 如果节点i还没有被遍历
        if i not in visited:
            # 对该节点进行广度优先搜索
            dfs(isConnected, i, visited)
            # 计数加一
            count += 1
    return count

In [ ]:
"""
417 Pacific Atlantic Water Flow
给定一个二维的非负整数矩阵，每个位置的值表示海拔高度。
假设左边和上百年是太平洋，右边和下边是大西洋，求从哪些位置向下流水，可以流到太平洋和大西洋。
水只能从海拔高的位置流到海拔低或相同的位置。
"""
# 输入是一个二维的非负整数数组，表示海拔高度
# 输出是一个二维的数组，其中第二个维度大小固定为2，表示满足条件的位置坐标

direction = [-1, 0, 1, 0, -1]

def dfs(heights: List[List[int]], can_reach: List[List[int]], r: int, c: int):
    if can_reach[r][c]:
        return
    can_reach[r][c] = True
    for i in range(4):
        x, y = r + direction[i], c + direction[i+1]
        if (x >= 0 and x < len(heights) and y >= 0 and y < len(heights[0]) and
            heights[x][y] >= heights[r][c]):
            dfs(heights, can_reach, x, y)

def pacificAtlantic(heights: List[List[int]]) -> List[List[int]]:
    m, n = len(heights), len(heights[0])
    can_reach_p = [[False for _ in range(n)] for _ in range(m)]
    can_reach_a = [[False for _ in range(n)] for _ in range(m)]
    for i in range(m):
        dfs(heights, can_reach_p, i, 0)
        dfs(heights, can_reach_a, i, n - 1)
    for j in range(n):
        dfs(heights, can_reach_p, 0, j)
        dfs(heights, can_reach_a, m - 1, j)
    return [
        [i, j] for i in range(m) for j in range(n)
        if can_reach_p[i][j] and can_reach_a[i][j]
    ]

```markdown
## 一、题目再翻译

给定一个 `m × n` 的非负整数矩阵 `heights`，每个元素代表该格子的**海拔**。

- **太平洋** 固定在**左边界 + 上边界**外侧。  
- **大西洋** 固定在**右边界 + 下边界**外侧。  

水只能从**高海拔**流向**低海拔或相等海拔**（四连通：上下左右）。  
问：**哪些格子**的水**既能流向太平洋，也能流向大西洋**？  
返回这些格子的坐标列表，顺序任意。

---

## 二、关键观察 —— “逆流”或“倒灌”思想

正向思维：从每个格子出发，看水能否流到两大洋 → 超时（`O((m·n)^2)`）。

反向思维：**从两大洋边界出发，逆流而上**（只能走**更高或相等**海拔），能走到的格子就是“可以流向该大洋”的格子。

- 对太平洋建一个 `can_reach_p` 矩阵，标记能逆流走到太平洋的格子。  
- 对大西洋建一个 `can_reach_a` 矩阵，标记能逆流走到大西洋的格子。  
- 最后遍历全图，两个矩阵同为 `True` 的格子即为答案。

时间复杂度：`O(m·n)`，每个格子最多被访问两次。

---

## 三、方向数组 & 辅助函数

```python
direction = [-1, 0, 1, 0, -1]
```

同样利用“错位”技巧，循环 4 次即可得到四连通偏移量：

| i | dx = direction[i] | dy = direction[i+1] |
|---|-------------------|---------------------|
| 0 | -1                | 0                   | 上 |
| 1 | 0                 | 1                   | 右 |
| 2 | 1                 | 0                   | 下 |
| 3 | 0                 | -1                  | 左 |

---

## 四、DFS 辅助函数逐行解析

```python
def dfs(heights, can_reach, r, c):
```

- `heights`：海拔矩阵（只读）  
- `can_reach`：当前大洋的访问标记矩阵  
- `(r, c)`：本次要标记的坐标  

```python
    if can_reach[r][c]:
        return
```

- 已经标记过，直接剪枝，避免重复搜索。

```python
    can_reach[r][c] = True
```

- 标记“该格子可以流到当前大洋”。

```python
    for i in range(4):
        x, y = r + direction[i], c + direction[i+1]
        if (x >= 0 and x < len(heights) and y >= 0 and y < len(heights[0]) and
            heights[x][y] >= heights[r][c]):
            dfs(heights, can_reach, x, y)
```

- 四向扩散，但**只能往“更高或相等”海拔走**（逆流）。  
- 越界跳过。  
- 满足条件就继续递归。

> 注意：这里是**反向爬高**，与“水往低处流”正好相反。

---

## 五、主函数 `pacificAtlantic` 逐行讲解

```python
def pacificAtlantic(heights):
    m, n = len(heights), len(heights[0])
```

- 取行列数。

```python
    can_reach_p = [[False for _ in range(n)] for _ in range(m)]
    can_reach_a = [[False for _ in range(n)] for _ in range(m)]
```

- 建立两大洋的访问标记矩阵，初始全 `False`。

### 1. 从太平洋边界出发

太平洋边界 = **第 0 列（左边界）** + **第 0 行（上边界）**

```python
    for i in range(m):
        dfs(heights, can_reach_p, i, 0)      # 左边界
    for j in range(n):
        dfs(heights, can_reach_p, 0, j)      # 上边界
```

### 2. 从大西洋边界出发

大西洋边界 = **第 n-1 列（右边界）** + **第 m-1 行（下边界）**

```python
    for i in range(m):
        dfs(heights, can_reach_a, i, n - 1)  # 右边界
    for j in range(n):
        dfs(heights, can_reach_a, m - 1, j)  # 下边界
```

### 3. 收集答案

```python
    return [
        [i, j] for i in range(m) for j in range(n)
        if can_reach_p[i][j] and can_reach_a[i][j]
    ]
```

- 列表推导式：同时被两大洋标记为 `True` 的格子即为所求。

---

## 六、复杂度分析

| 维度 | 复杂度 | 说明 |
|------|--------|------|
| 时间 | `O(m · n)` | 每个格子最多被访问 2 次（太平洋一次 + 大西洋一次） |
| 空间 | `O(m · n)` | 两个 `bool` 矩阵 + 递归栈最坏 `O(m · n)` |

> 若担心栈深度，可把 `dfs` 换成显式栈（迭代 DFS）或队列（BFS），复杂度不变。

---

## 七、易错点 & 常见疑问

1. **为什么“逆流”是 `>=` 而不是 `<=`？**  
   我们反向走：从大洋边界往高处爬，只能爬更高或相等海拔，才能覆盖所有“能流到大洋”的格子。

2. **边界循环会不会重复调用？**  
   左上角 `(0,0)` 会被左边界循环和上边界循环各访问一次，但第一次就标记 `True`，第二次直接 `return`，无性能问题。

3. **可以改成 BFS 吗？**  
   当然可以，把 `dfs` 换成队列 `popleft` 即可，复杂度同阶。

4. **方向数组能不能用 `[[-1,0],[0,1],[1,0],[0,-1]]`？**  
   完全没问题，可读性反而更高。

---

## 八、一行记忆口诀

> **“两洋边界倒着爬，能爬到的就是家；左右双表一比对，交集坐标全拿下。”**
```

下面把 `can_reach` 和 **整个 DFS 的细节** 拆开揉碎讲，保证你能在纸上画出每一步。

------------------------------------------------
一、先忘掉“水往低处流”，先学会“倒着爬”

题目说：  
水只能从 **高** → **低或相等** 流进大洋。

反过来，我们**从大洋边界出发**，只能走 **更高或相等** 的格子，就能**覆盖所有“能流向该大洋”的格子**。  
（因为只要有一条不下降的路径，水就能顺着它流回来。）

------------------------------------------------
二、`can_reach` 到底是啥

1. 它是 **二维布尔数组**，大小和地图一样。  
2. 含义：  
   `can_reach_p[i][j] == True`  **⇔** 格子 `(i,j)` 的水**可以流到太平洋**。  
   `can_reach_a[i][j] == True`  **⇔** 格子 `(i,j)` 的水**可以流到大西洋**。

3. 初始全是 `False`，**DFS 负责把能爬到的格子改成 True**。

------------------------------------------------
三、DFS 的 4 行核心到底做了什么

```python
def dfs(h, can, r, c):
    if can[r][c]: return      # ① 已经来过，直接回去
    can[r][c] = True          # ② 标记“我到过”
    for i in 4个方向:
        (x,y) = 新坐标
        if 不越界 且 新格子海拔 >= 当前海拔:   # ③ 只能“爬高或平”
            dfs(h, can, x, y)                # ④ 继续爬
```

- **① 剪枝**：同一块地儿别反复爬。  
- **② 占地儿**：告诉后面的人“我能流到大海”。  
- **③ 筛选**：逆流只能往**高或平**走，否则水回不来。  
- **④ 扩散**：把邻居也拉进来一起占地儿。

------------------------------------------------
四、边界值到底怎么“处理”

**没有特殊 `if` 判断**，越界直接在外层 `for` 循环里**天然避开**：

| 边界 | 循环起点 | 固定坐标 | 越界情况 |
|------|----------|----------|----------|
| 左边界 | `i` 从 `0~m-1` | `(i,0)` | 邻居 `y=-1` 时③条件自动失败 |
| 右边界 | `i` 从 `0~m-1` | `(i,n-1)` | 邻居 `y=n` 自动失败 |
| 上边界 | `j` 从 `0~n-1` | `(0,j)` | 邻居 `x=-1` 自动失败 |
| 下边界 | `j` 从 `0~n-1` | `(m-1,j)` | 邻居 `x=m` 自动失败 |

👉 **越界坐标根本进不了递归**，所以代码里不需要再写 `if x<0 or x>=m ... return`。

------------------------------------------------
五、一个 3×3 小例子，手动跑一遍

矩阵（数字=海拔）：

```
太平洋
↑
3 3 3
2 2 2
1 1 1 ← 太平洋
```

右边缘和下边缘是大西洋。  
我们从**太平洋边界**（第 0 列 + 第 0 行）开始爬：

1. 起点 `(0,0)` 海拔 3  
   - 标记 `can_p[0][0]=True`  
   - 四个方向：右 `(0,1)` 也是 3，满足 `>=`，递归过去  
   - 右 → `(0,2)` 也是 3，继续  
   - 这样第一行全被标记 `True`

2. 左边界 `(1,0)` 海拔 2  
   - 上邻居 `(0,0)` 海拔 3 ≥ 2，可以爬，但 `(0,0)` 已经 `True`，直接剪枝  
   - 右邻居 `(1,1)` 海拔 2 ≥ 2，继续爬 …  
   - 最终第二行也全 `True`

3. 第三行同理，**整个矩阵都被太平洋标记为 `True`**  
   （因为海拔单调不增，水一定能流回去）

对大西洋同样倒爬，会发现**整个矩阵也能被大西洋标记**。

最终答案：所有 9 个格子都满足，返回它们坐标即可。

------------------------------------------------
六、再总结一句话

> `can_reach` 就是“**倒爬地图**”里的**打卡机**：  
> DFS 每踩过一个格子，就打卡一次，  
> 最后只要**两个打卡机都盖过章**的格子，就是“能同时流向两大洋”的位置。

这样，**边界不用特殊处理**，**越界自动被循环条件挡住**，  
**海拔判断用 `>=` 保证逆流正确**，  
**布尔数组去重又记录结果**，一套操作浑然一体。

希望现在你再回头看代码，会觉得每一行都“理所应当”！
```

下面把「太平洋 ~ ~ ~ ~ ~ ... 大西洋」这张图重新整理成**可运行的整数矩阵**，并给出**完整手绘倒爬过程**（DFS 只爬高或平），让你一眼看懂 `can_reach_p` 与 `can_reach_a` 是怎么被标记的。

------------------------------------------------
一、把“图”变成矩阵

去掉装饰性边框，`~` 当作海水不看，括号只是突出，真正数字如下：

```
行0:  1  2  2  3  5
行1:  3  2  3  4  4
行2:  2  4  5  3  1
行3:  6  7  1  4  5
行4:  5  1  1  2  4
```

`m = 5`, `n = 5`  
太平洋：左边界 `(i,0)` + 上边界 `(0,j)`  
大西洋：右边界 `(i,4)` + 下边界 `(4,j)`

------------------------------------------------
二、建立两个“打卡机”

```
can_reach_p     初始全 False
can_reach_a     初始全 False
```

------------------------------------------------
三、从太平洋边界倒爬（DFS，只走≥当前海拔）

1. 左边界 `(i,0)` 依次启动  
   `(0,0)=1` → 把第一列能爬高的全打卡  
   `(1,0)=3` → 同上  
   `(2,0)=2`  
   `(3,0)=6`  
   `(4,0)=5`

2. 上边界 `(0,j)` 依次启动  
   `(0,1)=2` `(0,2)=2` `(0,3)=3` `(0,4)=5`  
   它们会把第一行横向连通，也会向下扩散。

**手绘结果**（T=True，F=False）

```
can_reach_p (太平洋)
  0 1 2 3 4
0 T T T T T
1 T T T T T
2 T T T F F
3 T T T F F
4 T T F F F
```

解释：  
- 右上角 `(0,4)` 海拔 5 最高，太平洋水一定能流过去，所以整行/整列被打通。  
- 右下角 `(2,4)=1` 比左边 `(2,3)=3` 低，**倒爬时过不去**，所以 `(2,4)` 保持 F。

------------------------------------------------
四、从大西洋边界倒爬

右边界 `(i,4)` + 下边界 `(4,j)` 启动，同理：

```
can_reach_a (大西洋)
  0 1 2 3 4
0 F F F T T
1 F F F T T
2 F F T T T
3 T T T T T
4 T T T T T
```

------------------------------------------------
五、求交集 → 答案坐标

把两张表叠在一起，同为 T 的格子留下：

```
(0,4) (0,3)
(1,4) (1,3)
(2,4) (2,3) (2,2)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)
```

按题目要求输出成 `List[List[int]]` 即可（顺序任意）：

```python
[
 [0, 3], [0, 4],
 [1, 3], [1, 4],
 [2, 2], [2, 3], [2, 4],
 [3, 0], [3, 1], [3, 2], [3, 3], [3, 4],
 [4, 0], [4, 1], [4, 2], [4, 3], [4, 4]
]
```

------------------------------------------------
六、对应代码运行验证

把上面矩阵直接喂给之前贴过的 `pacificAtlantic` 函数，返回的坐标列表与这里完全一致。

**结论**：  
- `can_reach_p` 与 `can_reach_a` 就是两张“倒爬地图”。  
- 边界值**没有特殊处理**，越界判断在③条件里自动失效。  
- 海拔 `>=` 保证“逆流”正确。  
- 最后**交集**即为所求。